# Evaluación de aceptación de MBE-Bot: resultados de los docentes

Con dos docentes no se hace análisis inferencial (Análisis de datos): las respuestas se reportan ítem por ítem y la percepción del módulo de analítica se considera **favorable si ambos puntúan ≥ 5 en los dos ítems de utilidad referidos a él**.

In [1]:
from pathlib import Path
import pandas as pd

DATOS = Path("/content/encuesta_docentes_seudonimizada.csv")
UMBRAL_CONTROL = 4    # se excluye si "Estoy respondiendo el cuestionario al azar" > 4

# variable -> (inicio del texto de la pregunta en el formulario, constructo, ¿ítem del módulo de analítica?)
ITEMS = {
    "pu_eficiencia_1a7":     ("Mis estudiantes aprenderían de forma más eficiente", "PU", False),
    "pu_identificar_1a7":    ("La herramienta me ayudaría a identificar los temas", "PU", True),
    "control_atencion_1a7":  ("Estoy respondiendo el cuestionario al azar", "Control", False),
    "pu_reporte_1a7":        ("El reporte de analítica para docentes me daría información útil", "PU", True),
    "pu_general_1a7":        ("En general, considero útil esta herramienta", "PU", False),
    "peou_aprender_1a7":     ("Aprender a usar esta herramienta me resultaría fácil", "PEOU", False),
    "peou_interaccion_1a7":  ("La forma de interactuar con la herramienta es clara y comprensible", "PEOU", False),
    "peou_estudiantes_1a7":  ("Sería fácil para mis estudiantes volverse hábiles", "PEOU", False),
    "peou_reporte_1a7":      ("El reporte para docentes es fácil de interpretar", "PEOU", True),
    "peou_general_1a7":      ("En general, encuentro fácil de usar esta herramienta", "PEOU", False),
}

ETIQUETAS = {   # texto de cada ítem en la tabla del artículo
    "pu_eficiencia_1a7": "Mis estudiantes aprenderían de forma más eficiente con la herramienta",
    "pu_identificar_1a7": "Me ayudaría a identificar los temas con más dificultades",
    "pu_reporte_1a7": "El reporte de analítica me daría información útil para decidir sobre mis clases",
    "pu_general_1a7": "En general, la considero útil para mi actividad docente",
    "peou_aprender_1a7": "Aprender a usarla me resultaría fácil",
    "peou_interaccion_1a7": "La forma de interactuar es clara y comprensible",
    "peou_estudiantes_1a7": "Sería fácil para mis estudiantes volverse hábiles usándola",
    "peou_reporte_1a7": "El reporte docente es fácil de interpretar sin conocimientos técnicos",
    "peou_general_1a7": "En general, la encuentro fácil de usar",
}

docentes = pd.read_csv(DATOS).set_index("id")
docentes

,pu_eficiencia_1a7,pu_identificar_1a7,control_atencion_1a7,pu_reporte_1a7,pu_general_1a7,peou_aprender_1a7,peou_interaccion_1a7,peou_estudiantes_1a7,peou_reporte_1a7,peou_general_1a7
id,,,,,,,,,,
D1,6,6,1,7,6,7,7,7,6,6
D2,5,6,1,6,6,7,6,6,6,6


## 1. Control de atención

Artículo, Resultados > Participantes: ninguno de los dos docentes fue excluido por el ítem de control de atención.

In [2]:
control = docentes["control_atencion_1a7"]
excluidos = control[control > UMBRAL_CONTROL].index.tolist()
print(f"Respuestas al ítem de control: {control.to_dict()}  ->  excluidos: {excluidos or 'ninguno'}")

Respuestas al ítem de control: {'D1': 1, 'D2': 1}  ->  excluidos: ninguno


## 2. Tabla de valoración de los docentes (escala 1–7)

Los ítems marcados con * se refieren al módulo de analítica docente.

In [3]:
filas = []
for var, (_, constructo, analitica) in ITEMS.items():
    if constructo == "Control":
        continue
    filas.append({"Constructo": constructo,
                  "Ítem": ETIQUETAS[var] + (" *" if analitica else ""),
                  "D1": docentes.loc["D1", var], "D2": docentes.loc["D2", var]})
tabla = pd.DataFrame(filas)
tabla

,Constructo,Ítem,D1,D2
0,PU,Mis estudiantes aprenderían de forma más efici...,6,5
1,PU,Me ayudaría a identificar los temas con más di...,6,6
2,PU,El reporte de analítica me daría información ú...,7,6
3,PU,"En general, la considero útil para mi activida...",6,6
4,PEOU,Aprender a usarla me resultaría fácil,7,7
5,PEOU,La forma de interactuar es clara y comprensible,7,6
6,PEOU,Sería fácil para mis estudiantes volverse hábi...,7,6
7,PEOU,El reporte docente es fácil de interpretar sin...,6,6
8,PEOU,"En general, la encuentro fácil de usar",6,6


## 3. Resumen que se reporta en el texto

In [4]:
puntajes = docentes.drop(columns="control_atencion_1a7")
pu_analitica = ["pu_identificar_1a7", "pu_reporte_1a7"]

minimo, maximo = int(puntajes.min().min()), int(puntajes.max().max())
favorable = bool((docentes[pu_analitica] >= 5).all().all())
rango_analitica = sorted(docentes[pu_analitica].stack().unique().tolist())

print(f"Rango de todas las puntuaciones: {minimo}-{maximo} (zona de acuerdo: 5-7)")
print(f"Ítems de utilidad del módulo de analítica: valores {rango_analitica}")
print(f"Percepción favorable del módulo (ambos >= 5 en los dos ítems): {favorable}")

Rango de todas las puntuaciones: 5-7 (zona de acuerdo: 5-7)
Ítems de utilidad del módulo de analítica: valores [6, 7]
Percepción favorable del módulo (ambos >= 5 en los dos ítems): True


## 4. Tabla de verificación frente al artículo

In [5]:
articulo = {   # valores de la tabla de docentes del artículo (D1, D2)
    "pu_eficiencia_1a7": (6, 5), "pu_identificar_1a7": (6, 6), "pu_reporte_1a7": (7, 6),
    "pu_general_1a7": (6, 6), "peou_aprender_1a7": (7, 7), "peou_interaccion_1a7": (7, 6),
    "peou_estudiantes_1a7": (7, 6), "peou_reporte_1a7": (6, 6), "peou_general_1a7": (6, 6),
}
verificacion = [(ETIQUETAS[v], f"{a[0]} / {a[1]}", f"{docentes.loc['D1', v]} / {docentes.loc['D2', v]}")
                for v, a in articulo.items()]
verificacion += [
    ("Docentes excluidos por control de atención", "0", str(len(excluidos))),
    ("Todas las puntuaciones en la zona de acuerdo (5-7)", "True", str(minimo >= 5 and maximo <= 7)),
    ("Ítems de analítica con puntuación 6 o 7", "True", str(set(rango_analitica) <= {6, 7})),
    ("Percepción favorable del módulo de analítica", "True", str(favorable)),
]
tabla_verif = pd.DataFrame(verificacion, columns=["Cifra", "Artículo", "Calculado"])
tabla_verif["Coincide"] = tabla_verif["Artículo"] == tabla_verif["Calculado"]
Path("resultados").mkdir(exist_ok=True)
tabla_verif.to_csv("resultados/verificacion_articulo_docentes.csv", index=False)
print(f"Coinciden {tabla_verif['Coincide'].sum()} de {len(tabla_verif)} cifras")
tabla_verif

Coinciden 13 de 13 cifras


,Cifra,Artículo,Calculado,Coincide
0,Mis estudiantes aprenderían de forma más efici...,6 / 5,6 / 5,True
1,Me ayudaría a identificar los temas con más di...,6 / 6,6 / 6,True
2,El reporte de analítica me daría información ú...,7 / 6,7 / 6,True
3,"En general, la considero útil para mi activida...",6 / 6,6 / 6,True
4,Aprender a usarla me resultaría fácil,7 / 7,7 / 7,True
5,La forma de interactuar es clara y comprensible,7 / 6,7 / 6,True
6,Sería fácil para mis estudiantes volverse hábi...,7 / 6,7 / 6,True
7,El reporte docente es fácil de interpretar sin...,6 / 6,6 / 6,True
8,"En general, la encuentro fácil de usar",6 / 6,6 / 6,True
9,Docentes excluidos por control de atención,0,0,True
